# 자동매매 데이터 수집 1단계

목표: KOSPI, KOSDAQ 시가총액 상위 30개 종목을 가져오고, 다음 단계에서 학습용 가격 데이터를 수집할 수 있게 종목 리스트를 저장합니다.

## 프로젝트 컨텍스트

- 목표: AI 모델을 학습시켜 주식 자동매매 프로그램 제작
- 초기 투자금: 약 200만 원
- 하루 목표 수익: 약 5,000원
- 우선순위: 데이터 수집, 백테스트, 모의투자, 리스크 관리
- 가상환경 이름: `autoasset`
- 현재 단계: KOSPI/KOSDAQ 시총 상위 종목과 일봉 데이터 수집

In [5]:
# 필요한 패키지가 설치되어 있는지 확인하는 구간입니다.
# FinanceDataReader는 한국 주식 종목 리스트와 가격 데이터를 가져올 때 사용합니다.
# 처음 실행할 때 FinanceDataReader가 없으면 아래 줄의 주석을 풀고 한 번만 실행하세요.


from pathlib import Path  # 폴더와 파일 경로를 운영체제에 맞게 다루기 위해 Path를 가져옵니다.
from datetime import datetime  # 오늘 날짜와 현재 시간을 만들기 위해 datetime을 가져옵니다.

import pandas as pd  # 표 형태의 데이터를 다루기 위해 pandas를 pd라는 이름으로 가져옵니다.
import FinanceDataReader as fdr  # 주식 데이터를 가져오기 위해 FinanceDataReader를 fdr이라는 이름으로 가져옵니다.

DATA_DIR = Path("data")  # 수집한 데이터를 저장할 data 폴더 경로를 만듭니다.
DATA_DIR.mkdir(exist_ok=True)  # data 폴더가 없으면 만들고, 이미 있으면 에러 없이 넘어갑니다.

TODAY = datetime.now().strftime("%Y%m%d")  # 현재 날짜를 20260715 같은 파일명용 문자열로 만듭니다.
TODAY  # 노트북 화면에 오늘 날짜 문자열을 출력합니다.

'20260715'

## 1. KOSPI, KOSDAQ 시총 상위 30개 가져오기

In [6]:
kospi_top30 = fdr.StockListing("KOSPI").copy()  # KOSPI 전체 종목 리스트를 가져온 뒤 원본 보호를 위해 복사합니다.
kospi_top30 = kospi_top30.sort_values("Marcap", ascending=False)  # KOSPI 종목을 시가총액이 큰 순서로 정렬합니다.
kospi_top30 = kospi_top30.head(30).reset_index(drop=True)  # 정렬된 결과에서 상위 30개만 남기고 인덱스를 0부터 다시 붙입니다.
kospi_top30.insert(0, "Rank", range(1, len(kospi_top30) + 1))  # 맨 앞 컬럼에 1위부터 시작하는 순위 값을 추가합니다.
kospi_top30 = kospi_top30[[  # 모델링과 확인에 필요한 컬럼만 골라서 다시 저장합니다.
    "Rank", "Code", "Name", "Market", "Close", "Changes", "ChagesRatio",  # 순위, 종목코드, 종목명, 시장, 종가, 전일대비, 등락률 컬럼입니다.
    "Volume", "Amount", "Marcap", "Stocks"  # 거래량, 거래대금, 시가총액, 상장주식수 컬럼입니다.
]]  # KOSPI 상위 30개 컬럼 선택을 마무리합니다.

kosdaq_top30 = fdr.StockListing("KOSDAQ").copy()  # KOSDAQ 전체 종목 리스트를 가져온 뒤 원본 보호를 위해 복사합니다.
kosdaq_top30 = kosdaq_top30.sort_values("Marcap", ascending=False)  # KOSDAQ 종목을 시가총액이 큰 순서로 정렬합니다.
kosdaq_top30 = kosdaq_top30.head(30).reset_index(drop=True)  # 정렬된 결과에서 상위 30개만 남기고 인덱스를 0부터 다시 붙입니다.
kosdaq_top30.insert(0, "Rank", range(1, len(kosdaq_top30) + 1))  # 맨 앞 컬럼에 1위부터 시작하는 순위 값을 추가합니다.
kosdaq_top30 = kosdaq_top30[[  # 모델링과 확인에 필요한 컬럼만 골라서 다시 저장합니다.
    "Rank", "Code", "Name", "Market", "Close", "Changes", "ChagesRatio",  # 순위, 종목코드, 종목명, 시장, 종가, 전일대비, 등락률 컬럼입니다.
    "Volume", "Amount", "Marcap", "Stocks"  # 거래량, 거래대금, 시가총액, 상장주식수 컬럼입니다.
]]  # KOSDAQ 상위 30개 컬럼 선택을 마무리합니다.

display(kospi_top30)  # KOSPI 상위 30개 결과를 노트북 화면에 표로 보여줍니다.
display(kosdaq_top30)  # KOSDAQ 상위 30개 결과를 노트북 화면에 표로 보여줍니다.

,Rank,Code,Name,Market,Close,Changes,ChagesRatio,Volume,Amount,Marcap,Stocks
0,1,005930,삼성전자,KOSPI,279500,16500,6.27,24246600,6777001899250,1629650161980000,5846278608
1,2,000660,SK하이닉스,KOSPI,2082000,169000,8.83,5960436,12609729711000,1490260645215000,712702365
2,3,402340,SK스퀘어,KOSPI,1382000,192000,16.13,1014638,1409660771000,182630406224000,131958386
3,4,005935,삼성전자우,KOSPI,192000,9800,5.38,3107995,603367428050,154697167938400,802371203
4,5,009150,삼성전기,KOSPI,1413000,153000,12.14,942256,1318434073000,105766273536000,74693696
5,6,005380,현대차,KOSPI,434000,9500,2.24,940211,407845046500,88660112678000,204757766
6,7,373220,LG에너지솔루션,KOSPI,335000,13000,4.04,323090,108516023750,78390000000000,234000000
7,8,032830,삼성생명,KOSPI,337500,20500,6.47,505008,170345360500,67400000000000,200000000
8,9,105560,KB금융,KOSPI,181600,1600,0.89,1769688,323092771750,64659573908200,354687734
9,10,207940,삼성바이오로직스,KOSPI,1383000,15000,1.10,40845,57093348000,64066676184000,46290951


,Rank,Code,Name,Market,Close,Changes,ChagesRatio,Volume,Amount,Marcap,Stocks
0,1,196170,알테오젠,KOSDAQ GLOBAL,288500,9000,3.22,694174,195898522250,15245314583500,53586343
1,2,247540,에코프로비엠,KOSDAQ GLOBAL,120900,8300,7.37,426280,51661379350,11866831644200,97830434
2,3,086520,에코프로,KOSDAQ GLOBAL,86400,7300,9.23,1216218,104967305300,11663171456800,135776152
3,4,036930,주성엔지니어링,KOSDAQ GLOBAL,209500,16900,8.77,2909363,608931083000,9761035410000,46481121
4,5,277810,레인보우로보틱스,KOSDAQ,430000,29500,7.37,63091,26829930750,8322539082000,19399858
5,6,240810,원익IPS,KOSDAQ GLOBAL,142700,15600,12.27,2366649,329458903650,7028814623200,49083901
6,7,950160,코오롱티슈진,KOSDAQ,77900,5000,6.86,324160,25297098450,6619175464000,84970160
7,8,058470,리노공업,KOSDAQ GLOBAL,77900,4800,6.57,717040,55217576550,5944524300000,76211850
8,9,319660,피에스케이,KOSDAQ,204500,1000,0.49,819229,165138031100,5880242942000,28966714
9,10,039030,이오테크닉스,KOSDAQ,385000,28000,7.84,130733,49557818750,4767665850000,12319550


## 2. 종목 리스트 저장하기

In [7]:
top60 = pd.concat([kospi_top30, kosdaq_top30], ignore_index=True)  # KOSPI 30개와 KOSDAQ 30개를 하나의 표로 합칩니다.

kospi_path = DATA_DIR / f"kospi_top30_marketcap_{TODAY}.csv"  # KOSPI 상위 30개 CSV 파일 경로를 만듭니다.
kosdaq_path = DATA_DIR / f"kosdaq_top30_marketcap_{TODAY}.csv"  # KOSDAQ 상위 30개 CSV 파일 경로를 만듭니다.
top60_path = DATA_DIR / f"krx_top60_marketcap_{TODAY}.csv"  # KOSPI와 KOSDAQ을 합친 상위 60개 CSV 파일 경로를 만듭니다.

kospi_top30.to_csv(kospi_path, index=False, encoding="utf-8-sig")  # KOSPI 상위 30개 표를 엑셀에서 한글이 잘 보이는 CSV로 저장합니다.
kosdaq_top30.to_csv(kosdaq_path, index=False, encoding="utf-8-sig")  # KOSDAQ 상위 30개 표를 엑셀에서 한글이 잘 보이는 CSV로 저장합니다.
top60.to_csv(top60_path, index=False, encoding="utf-8-sig")  # 상위 60개 통합 표를 엑셀에서 한글이 잘 보이는 CSV로 저장합니다.

print(kospi_path)  # 저장된 KOSPI CSV 파일 경로를 출력합니다.
print(kosdaq_path)  # 저장된 KOSDAQ CSV 파일 경로를 출력합니다.
print(top60_path)  # 저장된 통합 CSV 파일 경로를 출력합니다.

data\kospi_top30_marketcap_20260715.csv
data\kosdaq_top30_marketcap_20260715.csv
data\krx_top60_marketcap_20260715.csv


## 3. 다음 단계용: 상위 종목 일봉 데이터 수집

모델 학습에는 시총 순위만으로는 부족하므로, 상위 60개 종목의 일봉 OHLCV 데이터도 함께 저장합니다.

In [8]:
START_DATE = "2020-01-01"  # 일봉 데이터를 수집할 시작 날짜를 지정합니다.
END_DATE = datetime.now().strftime("%Y-%m-%d")  # 일봉 데이터를 수집할 종료 날짜를 오늘 날짜로 지정합니다.

price_dir = DATA_DIR / "daily_prices"  # 종목별 일봉 CSV를 저장할 하위 폴더 경로를 만듭니다.
price_dir.mkdir(exist_ok=True)  # daily_prices 폴더가 없으면 만들고, 이미 있으면 에러 없이 넘어갑니다.

price_frames = []  # 종목별 일봉 데이터를 하나로 합치기 위해 임시로 담아둘 리스트를 만듭니다.

for row in top60[["Code", "Name", "Market"]].itertuples(index=False):  # 상위 60개 종목에서 종목코드, 종목명, 시장을 한 줄씩 꺼내 반복합니다.
    code, name, market = row  # 현재 반복 중인 행에서 종목코드, 종목명, 시장 값을 각각 변수에 담습니다.
    try:  # 특정 종목 수집에 실패해도 전체 작업이 멈추지 않도록 예외 처리를 시작합니다.
        price = fdr.DataReader(code, START_DATE, END_DATE).reset_index()  # 해당 종목의 시작일~종료일 일봉 데이터를 가져오고 날짜 인덱스를 컬럼으로 바꿉니다.
        price.insert(0, "Code", code)  # 일봉 표 맨 앞에 종목코드 컬럼을 추가합니다.
        price.insert(1, "Name", name)  # 일봉 표 두 번째 위치에 종목명 컬럼을 추가합니다.
        price.insert(2, "Market", market)  # 일봉 표 세 번째 위치에 시장 구분 컬럼을 추가합니다.
        price_frames.append(price)  # 나중에 전체 종목 데이터를 합치기 위해 현재 종목의 일봉 표를 리스트에 추가합니다.

        file_path = price_dir / f"{market}_{code}_{name}.csv"  # 현재 종목의 일봉 데이터를 저장할 파일 경로를 만듭니다.
        price.to_csv(file_path, index=False, encoding="utf-8-sig")  # 현재 종목의 일봉 데이터를 CSV 파일로 저장합니다.
    except Exception as exc:  # 데이터 수집이나 저장 중 에러가 발생하면 여기로 넘어옵니다.
        print(f"수집 실패: {market} {code} {name} - {exc}")  # 실패한 종목과 에러 내용을 출력합니다.

daily_prices = pd.concat(price_frames, ignore_index=True)  # 리스트에 모아둔 모든 종목의 일봉 데이터를 하나의 큰 표로 합칩니다.
daily_prices_path = DATA_DIR / f"krx_top60_daily_prices_{TODAY}.csv"  # 통합 일봉 데이터 CSV 파일 경로를 만듭니다.
daily_prices.to_csv(daily_prices_path, index=False, encoding="utf-8-sig")  # 통합 일봉 데이터를 CSV 파일로 저장합니다.

print(daily_prices.shape)  # 통합 일봉 데이터의 행 개수와 컬럼 개수를 출력합니다.
print(daily_prices_path)  # 통합 일봉 데이터가 저장된 파일 경로를 출력합니다.
display(daily_prices.head())  # 통합 일봉 데이터의 앞부분 5줄을 화면에 표로 보여줍니다.

(91972, 10)
data\krx_top60_daily_prices_20260715.csv


,Code,Name,Market,Date,Open,High,Low,Close,Volume,Change
0,005930,삼성전자,KOSPI,2020-01-02,55500,56000,55000,55200,12993228,-0.010753
1,005930,삼성전자,KOSPI,2020-01-03,56000,56600,54900,55500,15422255,0.005435
2,005930,삼성전자,KOSPI,2020-01-06,54900,55600,54600,55500,10278951,0.000000
3,005930,삼성전자,KOSPI,2020-01-07,55700,56400,55600,55800,10009778,0.005405
4,005930,삼성전자,KOSPI,2020-01-08,56200,57400,55900,56800,23501171,0.017921


In [ ]:
추천 순서는 이거야.
일봉 데이터 확인
종목별 데이터가 제대로 들어왔는지
날짜가 비어 있는 종목이 있는지
Open, High, Low, Close, Volume 컬럼이 정상인지
2020년 이후 데이터 길이가 너무 짧은 종목이 있는지

기본 피처 생성
일간 수익률
5일/20일/60일 이동평균
거래량 이동평균
종가가 이동평균 위/아래인지
최근 5일, 20일 수익률
변동성

예측 타깃 생성
예를 들어:
내일 종가가 오늘보다 오르면 target = 1
내일 종가가 오늘보다 내리거나 같으면 target = 0

기준 전략 백테스트
AI 전에 단순 기준선을 만들어야 해.
예:
종가가 20일선 위
5일선이 20일선 위
거래량이 20일 평균 이상
다음 날 수익률을 확인

AI 모델 학습
처음은 LightGBM, XGBoost보다 더 단순하게:
Logistic Regression
Random Forest
정도부터 시작하면 좋아.

지금 바로 다음에 할 작업은 이거야.
상위 60개 종목의 일봉 데이터를 하나로 합친 뒤, 기본 피처 컬럼을 추가하기